# Experiment A: Sequential Text Processing & Vanilla RNN (PyTorch)

## 1. Objective
Implement a basic Elman **Recurrent Neural Network (RNN)** in PyTorch from scratch (`VanillaRNNClassifier`), train it on our leakage-free ArXiv scientific abstracts dataset, record training/validation losses, and compare against Majority-Class and TF-IDF + Logistic Regression baselines.

## 2. Model Architecture
```
Input Token IDs (B, T) -> nn.Embedding(V, 128, pad=0) -> Dropout(0.3)
  -> pack_padded_sequence -> nn.RNN(128, 128, nonlinearity='tanh')
  -> Pooling (Last Valid h_T  vs.  Masked Mean Pool)
  -> Dropout(0.3) -> Linear(128, 8 ArXiv Classes)
```

## 3. Mathematical Foundation
At each timestep $t \in \{1, \dots, T\}$, the hidden state $h_t \in \mathbb{R}^H$ updates sequentially:
$$h_t = \tanh(W_{ih} x_t + b_{ih} + W_{hh} h_{t-1} + b_{hh}), \quad h_0 = \mathbf{0}$$

During **Backpropagation Through Time (BPTT)**, gradients flow backward via repeated Jacobian products:
$$\frac{\partial \mathcal{L}}{\partial h_t} = \frac{\partial \mathcal{L}}{\partial h_T} \prod_{k=t+1}^{T} \text{diag}(1 - h_k^2) W_{hh}$$
Because $|1 - h_k^2| \le 1$, gradients decay exponentially across 200-word scientific abstracts (**vanishing gradients**), which we mitigate partly via gradient norm clipping (`max_norm=1.0`) and compare `last` vs. `mean` pooling.


In [ ]:
import json
from pathlib import Path
from scholarmind.core.config import load_config
from scholarmind.experiments.exp_a_rnn import run_experiment_a

cfg = load_config()
results = run_experiment_a(config=cfg, epochs=12, batch_size=32, seed=42)
print("Baselines:", json.dumps(results["baselines"], indent=2))
print("Vanilla RNN (last h_T) Test:", results["vanilla_rnn_last"]["test"])
print("Vanilla RNN (mean pool) Test:", results["vanilla_rnn_mean"]["test"])


In [ ]:
import pandas as pd
from IPython.display import Image, display

display(pd.DataFrame(results["vanilla_rnn_last"]["history"]))
display(Image(filename=results["loss_curve_plot"]))
